# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > Practice lab: Collaborative Filtering Recommender Systems

Trong exercise này, bạn sẽ triển khai collaborative filtering để xây dựng recommender system cho phim. 

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > Outline
- [ 1 - Notation](#1)
- [ 2 - Recommender Systems](#2)
- [ 3 - Movie ratings dataset](#3)
- [ 4 - Collaborative filtering learning algorithm](#4)
  - [ 4.1 Collaborative filtering cost function](#4.1)
    - [ Exercise 1](#ex01)
- [ 5 - Learning movie recommendations](#5)
- [ 6 - Recommendations](#6)
- [ 7 - Congratulations!](#7)


## Packages <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
Chúng ta sẽ sử dụng NumPy và Tensorflow Packages quen thuộc hiện nay.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - Ký hiệu


|Ký hiệu <br /> chung | Mô tả| Python (nếu có) |
|:-------------|:----------------------------------------------------------|||
| $r(i,j)$ | scalar; = 1 nếu user j xếp hạng movie i = 0 nếu không ||
| $y(i,j)$ | scalar; = xếp hạng do user j đưa ra trên movie i (nếu r(i,j) = 1 được xác định) ||
|$\mathbf{w}^{(j)}$ | vector; parameters dành cho user j ||
|$b^{(j)}$ |  scalar; parameter dành cho user j ||
| $\mathbf{x}^{(i)}$ |   vector; Xếp hạng feature cho phim i ||     
| $n_u$ | số lượng user |num_users|
| $n_m$ | số lượng phim | num_movies |
| $n$ | số lượng features | num_features |
| $\mathbf{X}$ |  matrix của vectors $\mathbf{x}^{(i)}$ | X |
| $\mathbf{W}$ |  matrix của vectors $\mathbf{w}^{(j)}$ | W |
| $\mathbf{b}$ |  vector của bias parameters $b^{(j)}$ | b |
| $\mathbf{R}$ | matrix của các phần tử $r(i,j)$ | R |


<a name="2"></a>
## 2 - Recommender Systems <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
Trong lab này, bạn sẽ triển khai thuật toán học collaborative filtering và áp dụng nó cho dataset xếp hạng phim.
Mục tiêu của collaborative filtering recommender system là tạo ra hai vectors: Đối với mỗi user, một 'parameter vector' thể hiện sở thích xem phim của user. Đối với mỗi phim, một feature vector có cùng kích thước thể hiện một số mô tả về phim. dot product của hai vectors cộng với bias term sẽ đưa ra ước tính về xếp hạng mà user có thể đưa ra cho bộ phim đó.

Sơ đồ bên dưới trình bày chi tiết cách học những vectors này.


<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>


Xếp hạng hiện tại được cung cấp ở dạng matrix như được hiển thị. $Y$ chứa xếp hạng; Bao gồm 0,5 đến 5 trong 0,5 bước. 0 nếu phim chưa được xếp hạng. $R$ có số 1 về xếp hạng phim. Phim xếp theo hàng, user xếp theo cột. Mỗi user có parameter vector $w^{user}$ và bias. Mỗi phim có một feature vector $x^{movie}$. Các vectors này được học đồng thời bằng cách sử dụng xếp hạng user/phim hiện có dưới dạng dữ liệu training. Một training example được hiển thị ở trên: $\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$. Điều đáng chú ý là feature vector $x^{movie}$ phải làm hài lòng tất cả user trong khi user vector $w^{user}$ phải làm hài lòng tất cả các bộ phim. Đây là nguồn gốc tên của phương pháp này - tất cả user cộng tác để tạo ra bộ xếp hạng.


<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>


Sau khi học được feature vectors và parameters, chúng có thể được sử dụng để dự đoán cách user có thể xếp hạng một bộ phim chưa được xếp hạng. Điều này được thể hiện trong sơ đồ trên. Phương trình này là một ví dụ về dự đoán xếp hạng cho user thứ nhất trên phim số 0.


Trong exercise này, bạn sẽ triển khai hàm `cofiCostFunc` tính toán collaborative filtering
hàm mục tiêu. Sau khi triển khai hàm mục tiêu, bạn sẽ sử dụng vòng lặp training tùy chỉnh TensorFlow để tìm hiểu parameters cho collaborative filtering. Bước đầu tiên là trình bày chi tiết tập dữ liệu và cấu trúc dữ liệu sẽ được sử dụng trong lab.


<a name="3"></a>
## 3 - Xếp hạng phim dataset <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
Tập dữ liệu được lấy từ [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) dataset.   
[F. Maxwell Harper và Joseph A. Konstan. 2015. MovieLens Datasets: Lịch sử và bối cảnh. Giao dịch ACM trên Hệ thống thông minh tương tác (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

dataset ban đầu có 9000 phim được 600 user đánh giá. dataset đã được giảm kích thước để tập trung vào các bộ phim từ những năm 2000. dataset này bao gồm các xếp hạng trên thang điểm từ 0,5 đến 5 với khoảng tăng 0,5 bước. dataset rút gọn có user $n_u = 443$ và phim $n_m= 4778$. 

Dưới đây, bạn sẽ tải phim dataset vào các biến $Y$ và $R$.

matrix $Y$ ($n_m \times n_u$ matrix) lưu trữ xếp hạng $y^{(i,j)}$. matrix $R$ là chỉ báo có giá trị nhị phân matrix, trong đó $R(i,j) = 1$ nếu user $j$ đưa ra xếp hạng cho phim $i$ và $R(i,j)=0$ nếu ngược lại. 

Trong suốt phần này của exercise, bạn cũng sẽ làm việc với
matrices, $\mathbf{X}$, $\mathbf{W}$ và $\mathbf{b}$: 

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

Hàng thứ $i$ của $\mathbf{X}$ tương ứng với
feature vector $x^{(i)}$ dành cho phim thứ $i$ và hàng thứ $j$
$\mathbf{W}$ tương ứng với một parameter vector $\mathbf{w}^{(j)}$, cho
User thứ $j$. Cả $x^{(i)}$ và $\mathbf{w}^{(j)}$ đều có chiều $n$
vectors. Vì mục đích của exercise này, bạn sẽ sử dụng $n=10$ và
do đó, $\mathbf{x}^{(i)}$ và $\mathbf{w}^{(j)}$ có 10 phần tử.
Tương ứng, $\mathbf{X}$ là một
$n_m \times 10$ matrix và $\mathbf{W}$ là $n_u \times 10$ matrix.

Chúng ta sẽ bắt đầu bằng cách tải xếp hạng phim dataset để hiểu cấu trúc của dữ liệu.
Chúng ta sẽ tải $Y$ và $R$ cùng với phim dataset.  
Chúng ta cũng sẽ tải $\mathbf{X}$, $\mathbf{W}$ và $\mathbf{b}$ với các giá trị được tính toán trước. Những giá trị này sẽ được tìm hiểu sau trong lab, nhưng chúng ta sẽ sử dụng các giá trị được tính toán trước để phát triển cost model.


In [2]:
# Tải dữ liệuX, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
# Từ matrix, chúng tôi có thể tính toán số liệu thống kê như xếp hạng trung bình.tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - Thuật toán học Collaborative filtering <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

Bây giờ, bạn sẽ bắt đầu triển khai quá trình học collaborative filtering
thuật toán. Bạn sẽ bắt đầu bằng cách thực hiện hàm mục tiêu. 

Thuật toán collaborative filtering trong bối cảnh phim
khuyến nghị xem xét một tập hợp parameter vectors chiều $n$
$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$, $\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ và $b^{(0)},...,b^{(n_u-1)}$, trong đó
model dự đoán xếp hạng cho phim $i$ bởi user $j$ là
$y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(i)}$ . Cho một dataset bao gồm
một tập hợp xếp hạng do một số user tạo ra trên một số phim, bạn muốn
tìm hiểu parameter vectors $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},
\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$  and $b^{(0)},...,b^{(n_u-1)}$ tạo ra sự phù hợp nhất (giảm thiểu
sai số bình phương).

Bạn sẽ hoàn thành code trong cofiCostFunc để tính cost
chức năng cho collaborative filtering.


<a name="4.1"></a>
### 4.1 Collaborative filtering cost function

collaborative filtering cost function được đưa ra bởi
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}$$
Tổng đầu tiên trong (1) là "với tất cả $i$, $j$ trong đó $r(i,j)$ bằng $1$" và có thể được viết:

$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

Bây giờ bạn nên viết cofiCostFunc (collaborative filtering cost function) để trả lại cost này.


<a name="ex01"></a>
### Exercise 1

**Đối với việc thực hiện vòng lặp:**   
Bắt đầu bằng cách triển khai cost function bằng vòng lặp for.
Hãy cân nhắc việc phát triển cost function theo hai bước. Đầu tiên, phát triển cost function mà không có regularization. Trường hợp test không bao gồm regularization được cung cấp bên dưới để kiểm tra việc triển khai của bạn. Sau khi thao tác này hoạt động, hãy thêm regularization và chạy test bao gồm regularization.  Lưu ý rằng bạn chỉ nên tích lũy cost cho user $j$ và phim $i$ nếu $R(i,j) = 1$.


In [40]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    # ## START CODE HERE ###    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += r * np.square((np.dot(w,x) + b_j - y ))
    J += (lambda_) * (np.sum(np.square(W)) + np.sum(np.square(X)))
    J = J/2
    # ## END CODE HERE ###
    return J

In [41]:
# Kiểm tra công khaifrom public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>Click để biết gợi ý</b></font></summary>
    Bạn có thể cấu trúc code thành hai vòng lặp for tương tự như phép tính tổng trong (1).   
    Trước tiên hãy triển khai code mà không cần regularization.   
    Lưu ý rằng một số phần tử trong (1) là vectors. Sử dụng np.dot(). Bạn cũng có thể sử dụng np.square().
    Hãy chú ý xem phần tử nào được lập chỉ mục bởi i và phần tử nào được lập chỉ mục bởi j. Đừng quên chia cho hai.
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    
<details>
    <summary><font size="2" color="darkblue"><b> Nhấp để biết thêm gợi ý</b></font></summary>
        
    Dưới đây là một số chi tiết hơn. Code bên dưới lấy từng phần tử ra khỏi matrix trước khi sử dụng. 
    Người ta cũng có thể tham khảo trực tiếp matrix.  
    Code này không chứa regularization.
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>Last Resort (triển khai hoàn toàn không chính quy)</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>regularization</b></font></summary>
     Regularization chỉ bình phương từng phần tử của W array và X array và chúng tính tổng tất cả các phần tử bình phương.
     Bạn có thể sử dụng np.square() và np.sum().

<details>
    Chi tiết <summary><font size="2" color="darkblue"><b>regularization</b></font></summary>
    
```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>


In [42]:
# Giảm kích thước tập dữ liệu để việc này chạy nhanh hơnnum_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Đánh giá cost functionJ = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**Đầu ra dự kiến (lambda = 0)**:  
$13.67$.


In [43]:
# Đánh giá cost function bằng regularizationJ = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**Đầu ra dự kiến**:

28.09


**Triển khai Vectorized**

Điều quan trọng là tạo một triển khai vectorized để tính toán $J$, vì sau này nó sẽ được gọi nhiều lần trong quá trình tối ưu hóa. Đại số tuyến tính được sử dụng không phải là trọng tâm của loạt bài này nên việc triển khai sẽ được cung cấp. Nếu bạn là chuyên gia về đại số tuyến tính, vui lòng tạo phiên bản của chúng ta mà không cần tham khảo code bên dưới. 

Chạy code bên dưới và xác minh rằng nó tạo ra kết quả tương tự như phiên bản không phải vectorized.


In [44]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [45]:
# Đánh giá cost functionJ = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Đánh giá cost function bằng regularizationJ = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**Đầu ra dự kiến**:  
Cost: 13,67  
Cost (với regularization): 28,09


<a name="5"></a>
## 5 - Học giới thiệu phim <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

Sau khi bạn triển khai xong collaborative filtering cost
chức năng, bạn có thể bắt đầu training thuật toán của chúng ta để thực hiện
gợi ý phim cho chính bạn. 

Trong ô bên dưới, bạn có thể nhập các lựa chọn phim của riêng mình. Thuật toán sau đó sẽ đưa ra khuyến nghị cho bạn! Chúng ta đã điền một số giá trị theo sở thích của chúng ta, nhưng sau khi mọi thứ phù hợp với lựa chọn của chúng ta, bạn nên thay đổi giá trị này để phù hợp với sở thích của chúng ta.
Danh sách tất cả các phim trong dataset có trong tệp [movie list](data/small_movie_list.csv).


In [46]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Kiểm tra tệp Small_movie_list.csv để biết id của từng phim trong dataset của chúng tôi# Ví dụ: Toy Story 3 (2010) có ID 2700 nên để đánh giá "5", bạn có thể đặtmy_ratings[2700] = 5 

# Hoặc giả sử bạn không thích Persuasion (2007), bạn có thể đặtmy_ratings[2609] = 2;

# Chúng tôi đã chọn một vài bộ phim chúng tôi thích / không thích và xếp hạng mà chúng tôi# đã cho như sau:my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


Bây giờ, hãy thêm những đánh giá này vào $Y$ và $R$ và bình thường hóa xếp hạng.


In [47]:
# Tải lại xếp hạng và thêm xếp hạng mớiY, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# Chuẩn hóa DatasetYnorm, Ymean = normalizeRatings(Y, R)

Hãy chuẩn bị training model. Khởi tạo parameters và chọn Adam optimizer.


In [48]:
# Giá trị hữu íchnum_movies, num_users = Y.shape
num_features = 100

# Đặt Parameters ban đầu (W, X), sử dụng tf.Variable để theo dõi các biến nàytf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Khởi tạo optimizer.optimizer = keras.optimizers.Adam(learning_rate=1e-1)

Bây giờ chúng ta hãy training collaborative filtering model. Điều này sẽ tìm hiểu parameters $\mathbf{X}$, $\mathbf{W}$ và $\mathbf{b}$.


Các hoạt động liên quan đến việc học $w$, $b$ và $x$ đồng thời không thuộc 'layers' điển hình được cung cấp trong TensorFlow neural network package.  Do đó, luồng được sử dụng trong Khóa 2: Model, Compile(), Fit(), Predict(), không được áp dụng trực tiếp. Thay vào đó, chúng ta có thể sử dụng vòng lặp training tùy chỉnh.

Nhớ rằng từ các lab trước đó các bước của gradient descent.
- Lặp lại cho đến khi hội tụ:
    - Tính toán chuyển tiếp
    - Tính toán đạo hàm của loss so với parameters
    - Cập nhật parameters bằng cách sử dụng learning rate và các dẫn xuất được tính toán 
    
TensorFlow có khả năng tính toán đạo hàm tuyệt vời cho bạn. Điều này được hiển thị dưới đây. Trong phần `tf.GradientTape()`, các hoạt động trên Biến Tensorflow được theo dõi. Khi `tape.gradient()` được gọi sau này, nó sẽ trả về gradient của loss tương ứng với các biến được theo dõi. Sau đó, gradients có thể được áp dụng cho parameters bằng optimizer. 
Đây là phần giới thiệu rất ngắn gọn về feature hữu ích của TensorFlow và các framework machine learning khác. Thông tin thêm có thể được tìm thấy bằng cách điều tra "các vòng lặp training tùy chỉnh" trong khuôn khổ quan tâm.


In [49]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Sử dụng GradientTape của TensorFlow    # để ghi lại các thao tác được sử dụng để tính toán cost    with tf.GradientTape() as tape:

        # Tính toán cost (chuyển tiếp có trong cost)        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Sử dụng băng gradient để tự động truy xuất    # gradients của các biến có thể huấn luyện đối với loss    grads = tape.gradient( cost_value, [X,W,b] )

    # Chạy một bước của gradient descent bằng cách cập nhật    # giá trị của các biến để giảm thiểu loss.    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Đăng nhập định kỳ.    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136168.7
Training loss at iteration 40: 51863.3
Training loss at iteration 60: 24598.8
Training loss at iteration 80: 13630.4
Training loss at iteration 100: 8487.6
Training loss at iteration 120: 5807.7
Training loss at iteration 140: 4311.6
Training loss at iteration 160: 3435.2
Training loss at iteration 180: 2902.1


<a name="6"></a>
## 6 - Khuyến nghị
Dưới đây, chúng ta tính toán xếp hạng cho tất cả phim và user, đồng thời hiển thị những phim được đề xuất. Những điều này dựa trên các phim và xếp hạng được nhập dưới dạng `my_ratings[]` ở trên. Để dự đoán xếp hạng của phim $i$ cho user $j$, bạn tính $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$. Điều này có thể được tính cho tất cả các xếp hạng bằng cách sử dụng phép nhân matrix.


In [50]:
# Tạo prediction bằng cách sử dụng weights và biases đã được huấn luyệnp = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

# khôi phục lại ý nghĩapm = p + Ymean

my_predictions = pm[:,0]

# sắp xếp predictionsix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 4.49 for movie My Sassy Girl (Yeopgijeogin geunyeo) (2001)
Predicting rating 4.48 for movie Martin Lawrence Live: Runteldat (2002)
Predicting rating 4.48 for movie Memento (2000)
Predicting rating 4.47 for movie Delirium (2014)
Predicting rating 4.47 for movie Laggies (2014)
Predicting rating 4.47 for movie One I Love, The (2014)
Predicting rating 4.46 for movie Particle Fever (2013)
Predicting rating 4.45 for movie Eichmann (2007)
Predicting rating 4.45 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.45 for movie Into the Abyss (2011)


Original vs Predicted ratings:

Original 5.0, Predicted 4.90 for Shrek (2001)
Original 5.0, Predicted 4.84 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.13 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predicted 4.88 for Harry Potter and the Chamber of Secrets (2002)
Original 5.0, Predic

Trong thực tế, thông tin bổ sung có thể được sử dụng để nâng cao predictions của chúng ta. Ở trên, xếp hạng dự đoán cho vài trăm bộ phim đầu tiên nằm trong một phạm vi nhỏ. Chúng ta có thể nâng cao những điều trên bằng cách chọn từ những bộ phim hàng đầu, những bộ phim có xếp hạng trung bình cao và những bộ phim có hơn 20 xếp hạng. Phần này sử dụng framework dữ liệu [Pandas](https://pandas.pydata.org/) có nhiều features sắp xếp tiện dụng.


In [51]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
1743,4.030965,4.252336,107,"Departed, The (2006)"
2112,3.985287,4.238255,149,"Dark Knight, The (2008)"
211,4.477792,4.122642,159,Memento (2000)
929,4.887053,4.118919,185,"Lord of the Rings: The Return of the King, The..."
2700,4.796530,4.109091,55,Toy Story 3 (2010)
653,4.357304,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
1122,4.004469,4.006494,77,Shaun of the Dead (2004)
1841,3.980647,4.000000,61,Hot Fuzz (2007)
3083,4.084633,3.993421,76,"Dark Knight Rises, The (2012)"
2804,4.434171,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...


<a name="7"></a>
## 7 - Congratulations! <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
Bạn đã triển khai recommender system hữu ích!
